# Example 4: GPTQ Quantization

GPTQ (Generative Pre-trained Transformer Quantization) is an advanced post-training quantization method that uses calibration data to minimize quantization error.

**Key Features:**
- Uses calibration data for optimal quantization
- Achieves better accuracy than naive quantization
- Fast inference (2-3× faster than FP16)
- Ideal for production deployment

In [ ]:
import torch
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: GPTQ Concept Demonstration

GPTQ minimizes the error: $||WX - W_qX||^2$

where:
- $W$ is the original weight matrix
- $W_q$ is the quantized weight matrix  
- $X$ is calibration data (activations)

**Key Insight:** Use calibration data to guide quantization, not just the weight distribution!

In [ ]:
# Create sample weight and calibration data
W = torch.randn(512, 512, device=device)
X = torch.randn(512, 1000, device=device)  # Calibration data

print(f"Weight matrix: {W.shape}")
print(f"Calibration data: {X.shape}")

# Original output
Y_original = W @ X

### Naive Quantization (Baseline)

In [ ]:
# Simple per-tensor quantization
scale_naive = W.abs().max() / 127
W_naive = torch.clamp(torch.round(W / scale_naive), -128, 127)
W_naive_dq = W_naive * scale_naive
Y_naive = W_naive_dq @ X

error_naive = torch.mean((Y_original - Y_naive) ** 2).item()
print(f"Naive quantization MSE: {error_naive:.6f}")

### GPTQ-style Quantization (Simplified)

Process columns sequentially and compensate errors using Hessian.

In [ ]:
# GPTQ-style quantization
W_gptq = W.clone()
W_gptq_quantized = torch.zeros_like(W)

# Compute Hessian (second-order info from calibration data)
H = 2 * X @ X.T / X.shape[1]
H_inv = torch.inverse(H + 1e-4 * torch.eye(H.shape[0], device=H.device))

# Quantize column by column with error compensation
for i in range(min(W.shape[1], 100)):  # Only first 100 columns for demo
    w_col = W_gptq[:, i]
    scale = w_col.abs().max() / 127
    if scale == 0:
        scale = 1.0
    
    w_q = torch.clamp(torch.round(w_col / scale), -128, 127) * scale
    W_gptq_quantized[:, i] = w_q
    
    # Compute error
    error = w_col - w_q
    
    # Compensate error in remaining columns
    if i < W.shape[1] - 1:
        compensation = torch.outer(H_inv[:, i] / H_inv[i, i], error)
        W_gptq[:, i+1:] -= compensation[:, None]

Y_gptq = W_gptq_quantized @ X
error_gptq = torch.mean((Y_original - Y_gptq) ** 2).item()

print(f"GPTQ-style quantization MSE: {error_gptq:.6f}")
print(f"\n✓ GPTQ reduced error by {(1 - error_gptq/error_naive)*100:.1f}%!")

## Part 2: GPTQ Quantization Process

### Steps to quantize a model with GPTQ:

1. **Prepare calibration dataset** (128-1024 samples)
2. **Configure quantization** (bits, group size)
3. **Run quantization** (layer-by-layer)
4. **Save quantized model**

### Example with auto-gptq:

```python
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

# Configure
quantize_config = BaseQuantizeConfig(
    bits=4,
    group_size=128,
    desc_act=False,
)

# Load and quantize
model = AutoGPTQForCausalLM.from_pretrained(
    model_name,
    quantize_config=quantize_config
)
model.quantize(calibration_dataset)

# Save
model.save_quantized("./gptq-model")
```

## Part 3: Quantization Methods Comparison

In [ ]:
import pandas as pd

comparison = {
    'Method': ['FP16', 'INT8 (Dynamic)', 'INT8 (bitsandbytes)', 'NF4', 'GPTQ', 'AWQ'],
    'Bits': [16, 8, 8, 4, 4, 4],
    'Memory': ['1.0x', '0.5x', '0.25x', '0.125x', '0.125x', '0.125x'],
    'Speed': ['1.0x', '1.5-2x', '1.2-1.5x', '1.0-1.2x', '2-3x', '2-3x'],
    'Accuracy': ['100%', '99-99.5%', '98-99%', '95-98%', '97-99%', '97-99%'],
    'Setup': ['Instant', 'Instant', 'Instant', 'Instant', '10-60min', '10-30min'],
    'Use Case': [
        'Max accuracy',
        'General inference',
        'Memory-constrained',
        'QLoRA fine-tuning',
        'Production (best speed)',
        'Production (alternative)'
    ]
}

df = pd.DataFrame(comparison)
print(df.to_string(index=False))

## Summary

### When to Use GPTQ:
- ✅ Production inference with strict latency requirements
- ✅ Need best 4-bit accuracy
- ✅ Can afford one-time quantization cost
- ✅ Have representative calibration data

### When to Use bitsandbytes Instead:
- ✅ Quick experimentation
- ✅ Fine-tuning with QLoRA
- ✅ No calibration data available
- ✅ Inference speed less critical

### Resources:
- Pre-quantized GPTQ models: https://huggingface.co/TheBloke
- Auto-GPTQ: https://github.com/PanQiWei/AutoGPTQ
- GPTQ paper: https://arxiv.org/abs/2210.17323